# Experiment 1: Uzbek NER checkpoint transfer

Полный самодостаточный цикл baseline для Kaggle: логика подготовки данных, обучения, инференса и exact-span оценки находится непосредственно в ячейках Notebook. Внешние Python-скрипты не запускаются.

Перед запуском включите `Settings -> Accelerator -> GPU` и `Internet -> On`, затем выполняйте ячейки сверху вниз. Результаты сохраняются в `/kaggle/working/artifacts`.

## 1. Зависимости

Notebook фиксирует PyTorch `2.6.0 + CUDA 12.4`: эта сборка совместима в том числе с Pascal GPU, которые может выдать Kaggle. Несовместимые `torchvision` и `torchaudio` удаляются — в NER pipeline они не используются. Transformers 4.x и совместимый TensorFlow/Keras нужны для однократной конвертации исходного `tf_model.h5` в PyTorch; итоговая обученная модель сохраняется как обычный PyTorch checkpoint.

Если ячейка сообщает, что PyTorch был заменён в уже запущенной сессии, выберите `Session -> Restart Session` и снова выполните все ячейки сверху вниз.

In [ ]:
import importlib.metadata as metadata
import subprocess
import sys

TORCH_VERSION = "2.6.0"
TORCH_INDEX_URL = "https://download.pytorch.org/whl/cu124"

try:
    installed_torch = metadata.version("torch")
except metadata.PackageNotFoundError:
    installed_torch = None

torch_was_loaded = "torch" in sys.modules
unused_torch_packages = []
for package in ("torchvision", "torchaudio"):
    try:
        metadata.version(package)
    except metadata.PackageNotFoundError:
        continue
    unused_torch_packages.append(package)

if unused_torch_packages:
    print("Removing unused incompatible packages:", ", ".join(unused_torch_packages))
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-q", "-y", *unused_torch_packages],
        check=True,
    )

if installed_torch is None or installed_torch.split("+", 1)[0] != TORCH_VERSION:
    print(
        f"Installing CUDA-compatible PyTorch {TORCH_VERSION} "
        f"instead of {installed_torch or 'missing'}..."
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-warn-conflicts",
            "--upgrade",
            "--force-reinstall",
            f"torch=={TORCH_VERSION}",
            "--index-url",
            TORCH_INDEX_URL,
        ],
        check=True,
    )
    if torch_was_loaded:
        raise RuntimeError(
            "PyTorch was replaced after it had already been imported. "
            "Restart the Kaggle session, then run all cells from the top."
        )
else:
    print("Compatible PyTorch is already installed:", installed_torch)

required = {
    "transformers": "4.57.1",
    "tokenizers": "0.22.1",
    "tensorflow": "2.18.0",
    "tf-keras": "2.18.0",
    "tqdm": "4.70.0",
}
to_install = []
for package, version in required.items():
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        to_install.append(f"{package}=={version}")

if to_install:
    print("Installing:", ", ".join(to_install))
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-warn-conflicts",
            *to_install,
        ],
        check=True,
    )
else:
    print("Other dependencies are already installed.")

In [ ]:
from pathlib import Path
import subprocess

REPOSITORY_URL = "https://github.com/koccyx/ner_uzb.git"
KAGGLE_REPO_DIR = Path("/kaggle/working/ner_uzb")
project_is_here = (Path.cwd() / "data/train.jsonl").exists()

if not project_is_here and Path("/kaggle/working").exists():
    if KAGGLE_REPO_DIR.exists() and not (KAGGLE_REPO_DIR / "data/train.jsonl").exists():
        raise RuntimeError(f"Incomplete project directory exists: {KAGGLE_REPO_DIR}")
    if not KAGGLE_REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPOSITORY_URL, str(KAGGLE_REPO_DIR)],
            check=True,
        )
        print("Repository cloned to:", KAGGLE_REPO_DIR)
    else:
        print("Repository already exists:", KAGGLE_REPO_DIR)
else:
    print("Using current project directory:", Path.cwd())

## 2. Проект и окружение

In [ ]:
import hashlib
import json
import math
import os
import platform
import random
import shutil
import time
from collections import Counter
from pathlib import Path
from typing import Any

import torch
from torch.nn.utils import clip_grad_norm_
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    get_linear_schedule_with_warmup,
)

candidates = [
    Path.cwd(),
    Path.cwd() / "ner_uzb",
    Path("/kaggle/working/ner_uzb"),
]
PROJECT_ROOT = next(
    (path.resolve() for path in candidates if (path / "data/train.jsonl").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project root not found. Clone ner_uzb into /kaggle/working first."
    )
os.chdir(PROJECT_ROOT)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Project:", PROJECT_ROOT)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("PyTorch CUDA runtime:", torch.version.cuda)
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU compute capability:", torch.cuda.get_device_capability(0))
    print("PyTorch CUDA architectures:", torch.cuda.get_arch_list())
    try:
        cuda_probe = torch.ones(1, device="cuda")
        cuda_probe.mul_(2)
        torch.cuda.synchronize()
        del cuda_probe
        print("CUDA kernel smoke test: OK")
    except RuntimeError as error:
        raise RuntimeError(
            "The installed PyTorch build cannot execute kernels on this GPU. "
            "Restart the Kaggle session and run the dependency cell first."
        ) from error

## 3. Конфигурация

`full` использует весь датасет. `medium` нужен для более короткого эксперимента. `smoke` проверяет только работоспособность и не предназначен для оценки качества.

Эксперимент стартует с узбекского NER-checkpoint `jmshd/roberta-ner-uz`. Он gated: сначала откройте страницу модели на Hugging Face, примите условия доступа, затем добавьте read-token в `Add-ons -> Secrets` под именем `HF_TOKEN` и включите доступ секрета для Notebook. Исходная NER-голова имеет несовместимые классы и будет заменена новой головой для `ORG/NAME/GEO`; fine-tuned XLM-R backbone сохранится.

In [ ]:
RUN_MODE = "full"  # smoke | medium | full
RUN_NAME = "uzbek_xlmr_transfer"
RUN_TRAIN = True
RUN_PREDICT = True
RUN_EVALUATE = True
OVERWRITE_OUTPUT = False
REQUIRE_CUDA = True

PROFILES = {
    "smoke": {"epochs": 2, "train_limit": 500, "dev_limit": 100},
    "medium": {"epochs": 3, "train_limit": 3000, "dev_limit": 300},
    "full": {"epochs": 3, "train_limit": None, "dev_limit": None},
}
MODEL_NAME = "jmshd/roberta-ner-uz"
MODEL_REVISION = "9284a1ffc39ebfa215d0b482305099a1a793c74c"
MODEL_FROM_TF = True
RESET_CLASSIFIER = True
MODEL_REQUIRES_AUTH = True
HF_TOKEN_SECRET_NAME = "HF_TOKEN"
TRAIN_BATCH_SIZE = 8
PREDICT_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0
MAX_LENGTH = 256
STRIDE = 64
SEED = 42
NUM_WORKERS = 2

if RUN_MODE not in PROFILES:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}")
if REQUIRE_CUDA and DEVICE != "cuda":
    raise RuntimeError(
        "CUDA is unavailable. Enable GPU in Kaggle Settings, then restart the session."
    )
if TRAIN_BATCH_SIZE < 1 or PREDICT_BATCH_SIZE < 1:
    raise ValueError("Batch sizes must be positive")
if GRADIENT_ACCUMULATION_STEPS < 1 or MAX_LENGTH < 1:
    raise ValueError("Gradient accumulation and max length must be positive")
if NUM_WORKERS < 0 or LEARNING_RATE <= 0 or WEIGHT_DECAY < 0:
    raise ValueError("Invalid optimizer or DataLoader configuration")
if not 0 <= WARMUP_RATIO < 1 or MAX_GRAD_NORM <= 0:
    raise ValueError("Invalid warmup ratio or max gradient norm")

HF_TOKEN = os.environ.get(HF_TOKEN_SECRET_NAME)
if MODEL_REQUIRES_AUTH and not HF_TOKEN and Path("/kaggle").exists():
    try:
        from kaggle_secrets import UserSecretsClient

        HF_TOKEN = UserSecretsClient().get_secret(HF_TOKEN_SECRET_NAME)
    except Exception as error:
        raise RuntimeError(
            f"Cannot read Kaggle Secret {HF_TOKEN_SECRET_NAME!r}. Accept access to "
            f"{MODEL_NAME!r}, add a Hugging Face read token to Kaggle Secrets, "
            "and enable this secret for the Notebook."
        ) from error
if MODEL_REQUIRES_AUTH and not HF_TOKEN:
    raise RuntimeError(
        f"Set the {HF_TOKEN_SECRET_NAME} environment variable before loading {MODEL_NAME!r}"
    )
if HF_TOKEN:
    from huggingface_hub import login

    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Hugging Face authentication: OK")

profile = PROFILES[RUN_MODE]
workspace = Path("/kaggle/working") if Path("/kaggle/working").exists() else PROJECT_ROOT
OUTPUT_DIR = workspace / "artifacts" / RUN_NAME
MODEL_DIR = OUTPUT_DIR / "model"
PREDICTIONS_PATH = OUTPUT_DIR / "dev_predictions.jsonl"
METRICS_PATH = OUTPUT_DIR / "dev_metrics.json"

print(json.dumps({
    "mode": RUN_MODE,
    "run_name": RUN_NAME,
    "device": DEVICE,
    "output_dir": str(OUTPUT_DIR),
    "model_name": MODEL_NAME,
    "model_revision": MODEL_REVISION,
    "model_from_tf": MODEL_FROM_TF,
    "reset_classifier": RESET_CLASSIFIER,
    **profile,
}, indent=2))

## 4. Проверка данных

In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def summarize_jsonl(path):
    records = 0
    entities = Counter()
    empty = 0
    with Path(path).open(encoding="utf-8") as stream:
        for line in stream:
            record = json.loads(line)
            records += 1
            empty += not record["entities"]
            entities.update(entity["label"] for entity in record["entities"])
    return {"records": records, "empty": empty, "entities": dict(entities)}


manifest = json.loads(Path("data/dataset_manifest.json").read_text(encoding="utf-8"))
for split_name in ("train", "dev"):
    split = manifest["splits"][split_name]
    actual_hash = sha256(split["path"])
    if actual_hash != split["sha256"]:
        raise ValueError(f"SHA-256 mismatch for {split['path']}")
    print(split_name, summarize_jsonl(split["path"]))
print("Dataset validation: OK")

## 5. Данные и BIO-разметка

В этой секции находится общая логика, которая раньше импортировалась из `baseline/common.py`: валидация JSONL, sliding windows, выравнивание span-разметки и BIO-декодирование.

In [ ]:
ENTITY_LABELS = ("ORG", "NAME", "GEO")
TAGS = (
    "O",
    "B-ORG", "I-ORG",
    "B-NAME", "I-NAME",
    "B-GEO", "I-GEO",
)
TAG_TO_ID = {tag: index for index, tag in enumerate(TAGS)}
JsonObject = dict[str, Any]
ModelFeature = dict[str, list[int]]
Offsets = list[tuple[int, int]]


def validate_entities(raw, text, source):
    if not isinstance(raw, list):
        raise ValueError(f"{source}: entities must be an array")
    entities = []
    seen = set()
    for index, entity in enumerate(raw):
        if not isinstance(entity, dict):
            raise ValueError(f"{source}/entities[{index}]: entity must be an object")
        label = entity.get("label")
        start = entity.get("start")
        end = entity.get("end")
        if label not in ENTITY_LABELS:
            raise ValueError(f"{source}/entities[{index}]: invalid label {label!r}")
        if (
            not isinstance(start, int) or isinstance(start, bool)
            or not isinstance(end, int) or isinstance(end, bool)
            or not 0 <= start < end <= len(text)
        ):
            raise ValueError(f"{source}/entities[{index}]: invalid offsets")
        key = (label, start, end)
        if key in seen:
            raise ValueError(f"{source}/entities[{index}]: duplicate entity")
        seen.add(key)
        entities.append({"label": label, "start": start, "end": end})

    entities.sort(key=lambda item: (item["start"], item["end"], item["label"]))
    for left, right in zip(entities, entities[1:]):
        if right["start"] < left["end"]:
            raise ValueError(f"{source}: overlapping entities are not supported")
    return entities


def read_records(path, *, require_entities, limit=None):
    path = Path(path)
    records = []
    seen_hashes = set()
    with path.open(encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, start=1):
            if not line.strip():
                raise ValueError(f"{path}:{line_number}: empty line")
            try:
                raw = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"{path}:{line_number}: invalid JSON: {error}") from error
            if not isinstance(raw, dict):
                raise ValueError(f"{path}:{line_number}: record must be an object")
            record_hash = raw.get("hash")
            text = raw.get("text")
            if not isinstance(record_hash, str) or not record_hash:
                raise ValueError(f"{path}:{line_number}: hash must be a non-empty string")
            if not isinstance(text, str):
                raise ValueError(f"{path}:{line_number}: text must be a string")
            if record_hash in seen_hashes:
                raise ValueError(f"{path}:{line_number}: duplicate hash {record_hash}")
            seen_hashes.add(record_hash)
            record = {"hash": record_hash, "text": text}
            if require_entities:
                record["entities"] = validate_entities(
                    raw.get("entities"), text, f"{path}:{line_number}"
                )
            records.append(record)
            if limit is not None and len(records) >= limit:
                break
    if not records:
        raise ValueError(f"{path}: no records")
    return records


def set_seed(seed, *, seed_cuda):
    random.seed(seed)
    torch.manual_seed(seed)
    if seed_cuda:
        torch.cuda.manual_seed_all(seed)


def load_fast_tokenizer(model_name_or_path):
    tokenizer_kwargs = {"use_fast": True}
    if not Path(model_name_or_path).exists():
        tokenizer_kwargs.update(revision=MODEL_REVISION, token=HF_TOKEN)
    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, **tokenizer_kwargs)
    if not tokenizer.is_fast:
        raise ValueError("A fast tokenizer with offset_mapping support is required")
    return tokenizer


def validate_window(tokenizer, max_length, stride):
    content_length = max_length - tokenizer.num_special_tokens_to_add(pair=False)
    if content_length < 1:
        raise ValueError("MAX_LENGTH is too small for tokenizer special tokens")
    if not 0 <= stride < content_length:
        raise ValueError(f"STRIDE must be between 0 and {content_length - 1}")


def tokenize_windows(tokenizer, text, *, max_length, stride):
    encoded = tokenizer(
        text,
        truncation=True,
        max_length=max_length,
        stride=stride,
        return_offsets_mapping=True,
        return_overflowing_tokens=True,
    )
    input_chunks = encoded["input_ids"]
    offset_chunks = encoded["offset_mapping"]
    if input_chunks and isinstance(input_chunks[0], int):
        input_chunks = [input_chunks]
        offset_chunks = [offset_chunks]

    windows = []
    for chunk_index, offsets in enumerate(offset_chunks):
        feature = {}
        for key in ("input_ids", "attention_mask"):
            if key not in encoded:
                continue
            values = encoded[key]
            feature[key] = values[chunk_index] if values and isinstance(values[0], list) else values
        windows.append((feature, [(int(start), int(end)) for start, end in offsets]))
    return windows


def align_labels(offsets, entities):
    labels = []
    entity_index = 0
    for start, end in offsets:
        if start == end:
            labels.append(-100)
            continue
        while entity_index < len(entities) and entities[entity_index]["end"] <= start:
            entity_index += 1
        if entity_index >= len(entities):
            labels.append(TAG_TO_ID["O"])
            continue
        entity = entities[entity_index]
        if end <= entity["start"] or start >= entity["end"]:
            labels.append(TAG_TO_ID["O"])
            continue
        prefix = "B" if start <= entity["start"] < end else "I"
        labels.append(TAG_TO_ID[f"{prefix}-{entity['label']}"])
    return labels


class TokenizedNerDataset(Dataset):
    def __init__(self, records, tokenizer, *, max_length, stride, description):
        self.features = []
        for record in tqdm(records, desc=description, unit="doc"):
            windows = tokenize_windows(
                tokenizer,
                record["text"],
                max_length=max_length,
                stride=stride,
            )
            for feature, offsets in windows:
                labels = align_labels(offsets, record["entities"])
                if not all(label == -100 for label in labels):
                    self.features.append({**feature, "labels": labels})
        if not self.features:
            raise ValueError(f"{description}: tokenization produced no trainable windows")

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index]


def decode_bio_tokens(tokens):
    entities = []
    current = None
    for start, end, tag in tokens:
        if tag == "O":
            if current is not None:
                entities.append(current)
                current = None
            continue
        prefix, separator, label = tag.partition("-")
        if separator != "-" or prefix not in {"B", "I"} or label not in ENTITY_LABELS:
            raise ValueError(f"Model returned unsupported tag {tag!r}")
        if prefix == "B" or current is None or current["label"] != label:
            if current is not None:
                entities.append(current)
            current = {"label": label, "start": start, "end": end}
        else:
            current["end"] = max(current["end"], end)
    if current is not None:
        entities.append(current)
    return entities

## 6. Обучение

Все функции обучения объявлены и вызываются в Notebook; CLI и `baseline/train.py` не используются.

In [ ]:
def prepare_output_dir(path, overwrite):
    path = Path(path)
    if path.exists() and any(path.iterdir()) and not overwrite:
        raise ValueError(
            f"Output directory is not empty: {path}; set OVERWRITE_OUTPUT=True to reuse it"
        )
    path.mkdir(parents=True, exist_ok=True)


def move_batch(batch, device):
    return {key: value.to(device) for key, value in batch.items()}


def loss_weight(batch):
    return int((batch["labels"] != -100).sum().item())


@torch.inference_mode()
def evaluate_loss(model, loader, device):
    model.eval()
    weighted_loss = 0.0
    token_count = 0
    for batch in tqdm(loader, desc="Dev loss", unit="batch", leave=False):
        batch = move_batch(batch, device)
        output = model(**batch)
        weight = loss_weight(batch)
        weighted_loss += float(output.loss.item()) * weight
        token_count += weight
    if not token_count:
        raise RuntimeError("Dev dataset contains no labeled tokens")
    return weighted_loss / token_count


def train_epoch(
    model,
    loader,
    optimizer,
    scheduler,
    device,
    *,
    gradient_accumulation_steps,
    max_grad_norm,
):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    weighted_loss = 0.0
    token_count = 0

    progress = tqdm(loader, desc="Train", unit="batch", leave=False)
    for batch_index, batch in enumerate(progress, start=1):
        batch = move_batch(batch, device)
        output = model(**batch)
        loss = output.loss
        group_start = ((batch_index - 1) // gradient_accumulation_steps) * gradient_accumulation_steps
        group_size = min(gradient_accumulation_steps, len(loader) - group_start)
        (loss / group_size).backward()

        weight = loss_weight(batch)
        weighted_loss += float(loss.detach().item()) * weight
        token_count += weight
        should_step = batch_index % gradient_accumulation_steps == 0 or batch_index == len(loader)
        if should_step:
            clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
        progress.set_postfix(loss=f"{loss.detach().item():.4f}")

    if not token_count:
        raise RuntimeError("Train dataset contains no labeled tokens")
    return weighted_loss / token_count


def save_model(model, tokenizer, model_dir, config):
    model_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(model_dir)
    tokenizer.save_pretrained(model_dir)
    (model_dir / "baseline_config.json").write_text(
        json.dumps(config, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )


def load_transfer_model():
    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        revision=MODEL_REVISION,
        token=HF_TOKEN,
        from_tf=MODEL_FROM_TF,
    )
    if RESET_CLASSIFIER:
        classifier = getattr(model, "classifier", None)
        if not isinstance(classifier, torch.nn.Linear):
            raise TypeError(
                f"Expected a linear token-classification head, got {type(classifier).__name__}"
            )
        new_classifier = torch.nn.Linear(
            classifier.in_features,
            len(TAGS),
            bias=classifier.bias is not None,
        )
        model._init_weights(new_classifier)
        model.classifier = new_classifier

    model.num_labels = len(TAGS)
    model.config.num_labels = len(TAGS)
    model.config.id2label = dict(enumerate(TAGS))
    model.config.label2id = {tag: index for index, tag in enumerate(TAGS)}
    return model


def train_model():
    output_dir = OUTPUT_DIR.resolve()
    prepare_output_dir(output_dir, OVERWRITE_OUTPUT)
    device = torch.device(DEVICE)
    set_seed(SEED, seed_cuda=device.type == "cuda")

    train_records = read_records(
        "data/train.jsonl",
        require_entities=True,
        limit=profile["train_limit"],
    )
    dev_records = read_records(
        "data/dev.jsonl",
        require_entities=True,
        limit=profile["dev_limit"],
    )
    tokenizer = load_fast_tokenizer(MODEL_NAME)
    validate_window(tokenizer, MAX_LENGTH, STRIDE)

    train_dataset = TokenizedNerDataset(
        train_records,
        tokenizer,
        max_length=MAX_LENGTH,
        stride=STRIDE,
        description="Tokenize train",
    )
    dev_dataset = TokenizedNerDataset(
        dev_records,
        tokenizer,
        max_length=MAX_LENGTH,
        stride=STRIDE,
        description="Tokenize dev",
    )
    collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True)
    generator = torch.Generator()
    generator.manual_seed(SEED)
    train_loader = DataLoader(
        train_dataset,
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        collate_fn=collator,
        num_workers=NUM_WORKERS,
        generator=generator,
    )
    dev_loader = DataLoader(
        dev_dataset,
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=False,
        collate_fn=collator,
        num_workers=NUM_WORKERS,
    )

    model = load_transfer_model().to(device)
    optimizer = AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    updates_per_epoch = math.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS)
    total_updates = updates_per_epoch * profile["epochs"]
    warmup_steps = int(total_updates * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_updates,
    )

    print(f"Device: {device}")
    print(f"Train: {len(train_records)} documents, {len(train_dataset)} windows")
    print(f"Dev: {len(dev_records)} documents, {len(dev_dataset)} windows")

    model_dir = output_dir / "model"
    history = []
    best_dev_loss = float("inf")
    baseline_config = {
        "schema_version": 1,
        "base_model": MODEL_NAME,
        "base_model_revision": MODEL_REVISION,
        "initialized_from_tf": MODEL_FROM_TF,
        "classifier_reset": RESET_CLASSIFIER,
        "tags": list(TAGS),
        "max_length": MAX_LENGTH,
        "stride": STRIDE,
        "seed": SEED,
    }
    for epoch in range(1, profile["epochs"] + 1):
        train_loss = train_epoch(
            model,
            train_loader,
            optimizer,
            scheduler,
            device,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
            max_grad_norm=MAX_GRAD_NORM,
        )
        dev_loss = evaluate_loss(model, dev_loader, device)
        history.append({"epoch": epoch, "train_loss": train_loss, "dev_loss": dev_loss})
        print(f"Epoch {epoch}: train_loss={train_loss:.6f}, dev_loss={dev_loss:.6f}")
        if dev_loss < best_dev_loss:
            best_dev_loss = dev_loss
            save_model(model, tokenizer, model_dir, baseline_config)

    run_summary = {
        **baseline_config,
        "train_records": len(train_records),
        "dev_records": len(dev_records),
        "train_windows": len(train_dataset),
        "dev_windows": len(dev_dataset),
        "epochs": profile["epochs"],
        "batch_size": TRAIN_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "best_dev_loss": best_dev_loss,
        "history": history,
    }
    (output_dir / "training_summary.json").write_text(
        json.dumps(run_summary, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    print(f"Best model: {model_dir}")
    return model_dir

In [ ]:
if RUN_TRAIN:
    started = time.monotonic()
    trained_model_dir = train_model()
    print(f"Training completed in {(time.monotonic() - started) / 60:.1f} min")
else:
    print("Training skipped; using:", MODEL_DIR)

## 7. Предсказания на полном dev

Инференс также выполняется функциями из Notebook, включая усреднение вероятностей токенов в перекрывающихся окнах.

In [ ]:
def read_baseline_config(model_dir):
    path = Path(model_dir) / "baseline_config.json"
    if not path.exists():
        return {}
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict):
        raise ValueError(f"{path}: expected a JSON object")
    return payload


def model_labels(model):
    labels = {int(index): str(label) for index, label in model.config.id2label.items()}
    if set(labels.values()) != set(TAGS) or set(labels) != set(range(len(TAGS))):
        raise ValueError(f"Model labels must be exactly {list(TAGS)}")
    return labels


def build_windows(records, tokenizer, *, max_length, stride):
    windows = []
    for record_index, record in enumerate(tqdm(records, desc="Tokenize", unit="doc")):
        for feature, offsets in tokenize_windows(
            tokenizer,
            record["text"],
            max_length=max_length,
            stride=stride,
        ):
            windows.append((record_index, feature, offsets))
    return windows


@torch.inference_mode()
def predict_token_scores(model, tokenizer, windows, record_count, *, batch_size, device):
    aggregated = [{} for _ in range(record_count)]
    model.eval()
    for batch_start in tqdm(
        range(0, len(windows), batch_size),
        desc="Predict",
        unit="batch",
    ):
        batch_windows = windows[batch_start:batch_start + batch_size]
        batch = tokenizer.pad(
            [feature for _, feature, _ in batch_windows],
            padding=True,
            return_tensors="pt",
        )
        batch = {key: value.to(device) for key, value in batch.items()}
        probabilities = torch.softmax(model(**batch).logits.float(), dim=-1).cpu()

        for row_index, (record_index, _, offsets) in enumerate(batch_windows):
            record_scores = aggregated[record_index]
            for token_index, (start, end) in enumerate(offsets):
                if start == end:
                    continue
                key = (start, end)
                score = probabilities[row_index, token_index]
                if key in record_scores:
                    previous, count = record_scores[key]
                    record_scores[key] = (previous + score, count + 1)
                else:
                    record_scores[key] = (score.clone(), 1)
    return aggregated


def decode_records(records, scores, id2label):
    predictions = []
    for record, record_scores in zip(records, scores):
        tagged_tokens = []
        for (start, end), (score_sum, count) in sorted(record_scores.items()):
            label_id = int((score_sum / count).argmax().item())
            tagged_tokens.append((start, end, id2label[label_id]))
        predictions.append({
            "hash": record["hash"],
            "entities": decode_bio_tokens(tagged_tokens),
        })
    return predictions


def write_jsonl(path, records):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as stream:
        for record in records:
            stream.write(json.dumps(record, ensure_ascii=False, separators=(",", ":")))
            stream.write("\n")


def predict_dev():
    if not MODEL_DIR.exists():
        raise FileNotFoundError(f"Model not found: {MODEL_DIR}")
    config = read_baseline_config(MODEL_DIR)
    max_length = int(config.get("max_length", MAX_LENGTH))
    stride = int(config.get("stride", STRIDE))
    device = torch.device(DEVICE)
    tokenizer = load_fast_tokenizer(str(MODEL_DIR))
    validate_window(tokenizer, max_length, stride)
    model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR).to(device)
    id2label = model_labels(model)

    records = read_records("data/dev.jsonl", require_entities=False)
    windows = build_windows(
        records,
        tokenizer,
        max_length=max_length,
        stride=stride,
    )
    scores = predict_token_scores(
        model,
        tokenizer,
        windows,
        len(records),
        batch_size=PREDICT_BATCH_SIZE,
        device=device,
    )
    predictions = decode_records(records, scores, id2label)
    write_jsonl(PREDICTIONS_PATH, predictions)
    print(f"Device: {device}")
    print(f"Records: {len(records)}, windows: {len(windows)}")
    print(f"Predictions: {PREDICTIONS_PATH}")
    return PREDICTIONS_PATH

In [ ]:
if RUN_PREDICT:
    started = time.monotonic()
    predictions_path = predict_dev()
    print(f"Prediction completed in {(time.monotonic() - started) / 60:.1f} min")
else:
    print("Prediction skipped; using:", PREDICTIONS_PATH)

## 8. Exact-span evaluation

Scorer встроен в Notebook и проверяет точное совпадение `hash`, `label`, `start` и `end`.

In [ ]:
LABELS = ("ORG", "NAME", "GEO")


def read_evaluation_jsonl(path, kind):
    path = Path(path)
    records = []
    seen_hashes = set()
    with path.open(encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, start=1):
            if not line.strip():
                raise ValueError(f"{path}:{line_number}: empty line")
            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"{path}:{line_number}: invalid JSON: {error}") from error
            if not isinstance(record, dict):
                raise ValueError(f"{path}:{line_number}: record must be an object")
            record_hash = record.get("hash")
            if not isinstance(record_hash, str) or not record_hash:
                raise ValueError(f"{path}:{line_number}: hash must be a non-empty string")
            if record_hash in seen_hashes:
                raise ValueError(f"{path}:{line_number}: duplicate hash {record_hash}")
            seen_hashes.add(record_hash)
            records.append(record)
    if not records:
        raise ValueError(f"{path}: no {kind} records")
    return records


def validate_evaluation_entities(raw, text_length, source):
    if not isinstance(raw, list):
        raise ValueError(f"{source}: entities must be an array")
    entities = set()
    for index, entity in enumerate(raw):
        if not isinstance(entity, dict):
            raise ValueError(f"{source}/entities[{index}]: entity must be an object")
        label = entity.get("label")
        start = entity.get("start")
        end = entity.get("end")
        if label not in LABELS:
            raise ValueError(f"{source}/entities[{index}]: invalid label {label!r}")
        if (
            not isinstance(start, int) or isinstance(start, bool)
            or not isinstance(end, int) or isinstance(end, bool)
            or not 0 <= start < end <= text_length
        ):
            raise ValueError(f"{source}/entities[{index}]: invalid offsets")
        key = (label, start, end)
        if key in entities:
            raise ValueError(f"{source}/entities[{index}]: duplicate entity")
        entities.add(key)
    return entities


def gold_by_hash(records, path):
    result = {}
    for index, record in enumerate(records, start=1):
        text = record.get("text")
        if not isinstance(text, str):
            raise ValueError(f"{path}:{index}: gold text must be a string")
        entities = validate_evaluation_entities(
            record.get("entities"), len(text), f"{path}:{index}"
        )
        result[record["hash"]] = {"text": text, "entities": entities}
    return result


def predictions_by_hash(records, path, gold):
    predicted_hashes = {record["hash"] for record in records}
    gold_hashes = set(gold)
    missing = sorted(gold_hashes - predicted_hashes)
    extra = sorted(predicted_hashes - gold_hashes)
    if missing or extra:
        raise ValueError(
            "Gold/prediction hashes differ: "
            f"missing={missing[:5]} ({len(missing)} total), "
            f"extra={extra[:5]} ({len(extra)} total)"
        )
    result = {}
    for index, record in enumerate(records, start=1):
        record_hash = record["hash"]
        gold_record = gold[record_hash]
        if "text" in record and record["text"] != gold_record["text"]:
            raise ValueError(f"{path}:{index}: prediction text differs from gold for {record_hash}")
        result[record_hash] = validate_evaluation_entities(
            record.get("entities"),
            len(gold_record["text"]),
            f"{path}:{index}",
        )
    return result


def metric_values(tp, fp, fn):
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "gold": tp + fn,
        "predicted": tp + fp,
    }


def calculate_metrics(gold, predictions):
    counts = {label: {"tp": 0, "fp": 0, "fn": 0} for label in LABELS}
    for record_hash, gold_record in gold.items():
        gold_entities = gold_record["entities"]
        predicted_entities = predictions[record_hash]
        for label in LABELS:
            gold_label = {entity for entity in gold_entities if entity[0] == label}
            predicted_label = {entity for entity in predicted_entities if entity[0] == label}
            counts[label]["tp"] += len(gold_label & predicted_label)
            counts[label]["fp"] += len(predicted_label - gold_label)
            counts[label]["fn"] += len(gold_label - predicted_label)

    by_label = {
        label: metric_values(values["tp"], values["fp"], values["fn"])
        for label, values in counts.items()
    }
    micro = metric_values(
        sum(values["tp"] for values in counts.values()),
        sum(values["fp"] for values in counts.values()),
        sum(values["fn"] for values in counts.values()),
    )
    macro = {
        metric: sum(by_label[label][metric] for label in LABELS) / len(LABELS)
        for metric in ("precision", "recall", "f1")
    }
    return {
        "schema_version": 1,
        "matching": "same hash and exact label/start/end",
        "records": len(gold),
        "by_label": by_label,
        "micro": micro,
        "macro": macro,
    }


def print_metrics(metrics):
    header = f"{'scope':<8} {'precision':>10} {'recall':>10} {'f1':>10} {'tp':>8} {'fp':>8} {'fn':>8}"
    print(header)
    print("-" * len(header))
    for label in LABELS:
        values = metrics["by_label"][label]
        print(
            f"{label:<8} {values['precision']:>10.4f} {values['recall']:>10.4f} "
            f"{values['f1']:>10.4f} {values['tp']:>8} {values['fp']:>8} {values['fn']:>8}"
        )
    micro = metrics["micro"]
    print(
        f"{'micro':<8} {micro['precision']:>10.4f} {micro['recall']:>10.4f} "
        f"{micro['f1']:>10.4f} {micro['tp']:>8} {micro['fp']:>8} {micro['fn']:>8}"
    )
    macro = metrics["macro"]
    print(
        f"{'macro':<8} {macro['precision']:>10.4f} {macro['recall']:>10.4f} "
        f"{macro['f1']:>10.4f} {'-':>8} {'-':>8} {'-':>8}"
    )


def evaluate_predictions():
    if not PREDICTIONS_PATH.exists():
        raise FileNotFoundError(f"Predictions not found: {PREDICTIONS_PATH}")
    gold_path = Path("data/dev.jsonl").resolve()
    predictions_path = PREDICTIONS_PATH.resolve()
    gold_records = read_evaluation_jsonl(gold_path, "gold")
    gold = gold_by_hash(gold_records, gold_path)
    predictions = predictions_by_hash(
        read_evaluation_jsonl(predictions_path, "prediction"),
        predictions_path,
        gold,
    )
    metrics = calculate_metrics(gold, predictions)
    print_metrics(metrics)
    METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
    METRICS_PATH.write_text(
        json.dumps(metrics, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    print(f"Metrics: {METRICS_PATH}")
    return metrics

In [ ]:
if RUN_EVALUATE:
    metrics = evaluate_predictions()
else:
    print("Evaluation skipped; using:", METRICS_PATH)

## 9. Результаты и архив

In [ ]:
if METRICS_PATH.exists():
    metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
    print("Micro F1:", f"{metrics['micro']['f1']:.4f}")
    print("Macro F1:", f"{metrics['macro']['f1']:.4f}")
    for label, values in metrics["by_label"].items():
        print(
            f"{label:>4}: P={values['precision']:.4f} "
            f"R={values['recall']:.4f} F1={values['f1']:.4f}"
        )

if not OUTPUT_DIR.exists():
    raise FileNotFoundError(f"Output directory not found: {OUTPUT_DIR}")
archive_path = shutil.make_archive(
    str(OUTPUT_DIR.parent / RUN_NAME),
    "gztar",
    root_dir=OUTPUT_DIR,
)
print("Artifacts:", OUTPUT_DIR)
print("Archive:", archive_path)